# NuCLS · Entregable 2 · Continuar semillas de los modelos base
**Andrés Duarte Chaves — 10 de noviembre de 2026**

Este cuaderno es nuevo e independiente. Completa el plan de 22 semillas de CNN-BN, DenseNet121 y ResNet50. No repite búsqueda ni piloto y **no evalúa la reserva**. El Entregable 1 permanece intacto.

**Antes de empezar:** selecciona GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. No ejecutes simultáneamente el notebook antiguo ni otra sesión sobre estos resultados. Ejecuta las celdas 1–4 en orden. La primera ejecución puede dedicar varios minutos a copiar los datos y verificar los checkpoints.

El cierre anterior verificó 6101–6111. Si hay corridas posteriores completas también se omiten. Las parciales con `ultimo.pt` se copian a la carpeta nueva y continúan desde la siguiente época. Solo se repite el trabajo de una época que no alcanzó a guardarse.

**Origen de solo lectura:** `NuCLS_V2_RESULTADOS_pool_corregido`.
**Salida nueva en Drive:** `NuCLS_E2_CONTINUACION_BASES_22`.
**Fuente de imágenes si se reinició Colab:** `NuCLS_V2_DATOS.zip`, en la raíz de Mi unidad.

Este archivo es un lanzador preparado, aún sin salidas GPU. Después de ejecutarlo, guarda una copia con las salidas para la entrega.

## 1. Conectar Drive y comprobar el entorno

In [ ]:
import os, sys, json
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')

ORIGEN = Path('/content/drive/MyDrive/NuCLS_V2_RESULTADOS_pool_corregido')
SALIDA = Path('/content/drive/MyDrive/NuCLS_E2_CONTINUACION_BASES_22')
CODIGO = Path('/content/nucls_e2_codigo')
DATOS = Path('/content/nucls_e2_datos')
ZIP_DATOS = Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
assert ORIGEN != SALIDA
assert (ORIGEN / 'configuracion.json').is_file(), 'No se encontró la carpeta original con checkpoints. Un ZIP de métricas no la sustituye.'
config_original = json.loads((ORIGEN / 'configuracion.json').read_text())
import torch, torchvision, numpy as np
actual = {'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__), 'numpy': np.__version__}
print('Entorno actual:', actual)
print('Entorno original:', {k: config_original[k] for k in actual})
assert torch.cuda.is_available(), 'Selecciona GPU en este NUEVO notebook antes de continuar.'
for k, v in actual.items():
    assert v == str(config_original[k]), f'Versión distinta de {k}. Copia el mensaje para revisar el entorno; no se instalará ni actualizará automáticamente.'
assert tuple(sys.version.split()[0].split('.')[:2]) == tuple(config_original['python'].split()[0].split('.')[:2]), 'Versión de Python incompatible con el origen.'
print('GPU:', torch.cuda.get_device_name())
print('Salida nueva:', SALIDA)


## 2. Descargar el código fijado y preparar solo train/validación

In [ ]:
import urllib.request, hashlib, subprocess
COMMIT = 'da26ec58e232d3079d6c69e21f7f0ef1af36ca53'
HASHES = {'modelos/core_v2.py': 'af55fd33279b0a02852e1d5cc6c154d369a918029defb258a2a805d3971ed8f5', 'modelos/candidatos_v2.py': '0bd4d7acd1fb00124b8c6f7bc37390c6ade4efd411f7e1d7bbe00d38efce0e10', 'modelos/experimento_v2.py': 'ef3ae760413768b1c6e6dd3aaa2a533064fbc2ea1d5e5fb0f2e719b7242a34e8', 'scripts/continuar_semillas.py': 'af61cb86d7555989d246c6c18cde9f975bf07130188cb59373262cf532497d81', 'scripts/preparar_datos.py': '22f703338d3a65f2d318bee2a852be0a6bfbbae69142166ddabbc9c39db061da', 'protocolo/integridad_motor.json': 'efd3383759e257252dc876d6f2bbec6dd026ff710c92ce16c123c40c5e0153da'}
BASE = f'https://raw.githubusercontent.com/aduartech81/NuCLS_DeepLearning_Entregable2/{COMMIT}/'
for relative, expected in HASHES.items():
    with urllib.request.urlopen(BASE + relative, timeout=90) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected, 'Descarga diferente: ' + relative
    target = CODIGO / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + '.descarga_tmp')
    tmp.write_bytes(data)
    tmp.replace(target)
print('Código verificado:', COMMIT)
# Se usa una carpeta local nueva. El dataset anterior no se modifica.
if not (DATOS / 'manifiesto.csv').exists():
    assert ZIP_DATOS.is_file(), 'Falta NuCLS_V2_DATOS.zip en Mi unidad; no volver a descargarlo si ya existe en otra carpeta de Drive.'
    subprocess.run([sys.executable, '-u', str(CODIGO / 'scripts/preparar_datos.py'),
                    '--zip', str(ZIP_DATOS), '--salida', str(DATOS)], check=True)
else:
    print('Carpeta de desarrollo E2 existente; el entrenador verificará sus imágenes antes de empezar.')
print('Preparación terminada. No se han extraído imágenes de reserva.')


## 3. Inventario antes de entrenar
Esta celda no entrena. Revisa las 66 combinaciones, contrasta los resultados y muestra `completa`, `reanudar` o `nueva`. Verificar hashes de checkpoints grandes desde Drive puede tardar varios minutos. Si un resultado está incompleto o no coincide, se detiene; no se borra ni reentrena automáticamente.

In [ ]:
COMANDO = [sys.executable, '-u', str(CODIGO / 'scripts/continuar_semillas.py'),
           '--origen', str(ORIGEN), '--salida', str(SALIDA), '--datos', str(DATOS)]
subprocess.run(COMANDO, check=True)
inventario = json.loads((SALIDA / 'inventario_corridas.json').read_text())
for estado in ['completa', 'reanudar', 'nueva']:
    print(estado.upper(), sum(r['estado'] == estado for r in inventario))
print('Si no hay errores, ejecuta la celda 4.')


## 4. Continuar únicamente las corridas pendientes
Puede durar varias horas. Se conservan checkpoints por época. Las primeras 33 corridas sumaron unas 11,12 horas de épocas con el hardware anterior: es una referencia, no una duración garantizada.

En caso de desconexión, primero comprueba si esta ejecución sigue activa. No abras otra sesión sobre la misma salida. Al reiniciar una sesión terminada, vuelve a ejecutar 1–4: los resultados completos se omitirán.

Si detienes la celda, se intentará detener también su proceso de entrenamiento. No cierres a la fuerza mientras aparezca el mensaje de detención.

In [ ]:
import datetime, signal
SALIDA.mkdir(parents=True, exist_ok=True)
log = SALIDA / ('continuacion_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.txt')
with log.open('a', buffering=1) as archivo_log:
    proceso = subprocess.Popen(COMANDO + ['--ejecutar'], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for linea in proceso.stdout:
            print(linea, end='', flush=True)
            archivo_log.write(linea)
        codigo_salida = proceso.wait()
    except KeyboardInterrupt:
        print('Deteniendo el proceso de esta celda; los checkpoints anteriores permanecen guardados.', flush=True)
        proceso.send_signal(signal.SIGINT)
        try:
            proceso.wait(timeout=60)
        except subprocess.TimeoutExpired:
            proceso.terminate()
            proceso.wait(timeout=30)
        raise
assert codigo_salida == 0, f'El proceso se detuvo con código {codigo_salida}. Revisa el mensaje; no borres las carpetas.'
print('Log guardado:', log)


## 5. Consultar el cierre de desarrollo y ubicar el ZIP

In [ ]:
estado = SALIDA / 'estado_continuacion.json'
if estado.exists():
    print(estado.read_text())
else:
    print('Aún no hay cierre de 22. Los resultados completos y checkpoints quedan guardados.')
print('Índice de las corridas:', SALIDA / 'indice_validacion.csv')
zip_resultados = SALIDA.parent / (SALIDA.name + '_RESULTADOS.zip')
if zip_resultados.exists():
    print('ZIP de evidencias nuevas (sin pesos):', zip_resultados)
print('Los resultados antiguos se referencian en el índice; permanecen en la carpeta original.')
print('Guarda este notebook con sus salidas: Archivo → Descargar → Descargar .ipynb.')


## Recuperación excepcional de un bloqueo
Solo si el proceso anterior ya terminó y el programa informa `EJECUCION_ACTIVA.json`. Una pestaña desconectada no demuestra que haya terminado. La siguiente celda **no hace nada** mientras la variable esté en `False`. Tras verificar que no hay otro proceso, cambiarla a `True`, ejecutarla una vez y volver a las celdas 3–4. Si el proceso sigue activo en el mismo equipo, el script rechaza retirar el bloqueo.

In [ ]:
CONFIRMO_QUE_NO_HAY_OTRA_EJECUCION = False
if CONFIRMO_QUE_NO_HAY_OTRA_EJECUCION:
    subprocess.run(COMANDO + ['--liberar-bloqueo'], check=True)
else:
    print('No se modifica ningún bloqueo.')
